# Classificação de Séries Temporais — Template

Este notebook mostra, de ponta a ponta, como treinar e comparar classificadores que detectam **períodos de falha** em dados multivariados de sensores (uma amostra por minuto, uma coluna por variável).

O fluxo é:

1. **Carregar** os dados (CSV próprio ou um dataset sintético de exemplo);
2. **Pré-processar**: remover períodos com o equipamento desligado e rotular os períodos de falha;
3. **Dividir** em treino e teste respeitando a ordem temporal;
4. **Janelar** as séries (ex.: janelas de 60 min com passo de 30 min);
5. Treinar três famílias de modelos:
   - **Features estatísticas + ML clássico** (Random Forest, XGBoost, kNN);
   - **ROCKET / MiniROCKET** (kernels convolucionais aleatórios + classificador linear);
   - **Deep learning em PyTorch** (CNN 1D, InceptionTime, ResNet 1D e ResNet 2D);
6. **Comparar** os resultados no conjunto de teste.

> **Para usar com seus dados:** altere apenas a célula de **Configuração** (seção 2). Com `USE_SYNTHETIC_DATA = True` o notebook roda sem nenhum arquivo externo.

## 1. Imports

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import RidgeClassifierCV
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sktime.transformations.panel.rocket import MiniRocketMultivariate, Rocket
from xgboost import XGBClassifier

from tsc.data import label_periods, load_dataset, make_synthetic_dataset, remove_off_periods, select_period
from tsc.evaluation import classification_metrics, plot_history, plot_predictions, plot_roc_curves, plot_signal
from tsc.features import extract_window_features
from tsc.models import MODELS
from tsc.training import get_device, predict_proba, train_model
from tsc.windowing import fit_window_scaler, make_windows, scale_windows, temporal_split

warnings.filterwarnings("ignore")
print("Dispositivo para deep learning:", get_device())

## 2. Configuração

Todos os parâmetros que dependem do dataset ficam aqui. Os valores padrão funcionam com o dataset sintético.

In [ ]:
# --- Fonte dos dados ----------------------------------------------------------
# True: gera um dataset sintético (útil para conhecer o pipeline sem dados reais)
# False: lê o CSV em DATA_PATH (separador ";", uma coluna de timestamp + uma coluna por variável)
USE_SYNTHETIC_DATA = True
DATA_PATH = Path("../data/meu_dataset.csv")
TIMESTAMP_COL = "Timestamp"
LABEL_COL = "Falhas"
DROP_COLUMNS = []          # colunas constantes, com muitos NaN ou irrelevantes para a análise
SAMPLING_FREQ = "1min"     # período de amostragem dos dados
PLOT_VARIABLE = "pressao"  # variável exibida no gráfico exploratório

# --- Períodos de falha (rótulos) ----------------------------------------------
# Lista de (início, fim). Com dados sintéticos é substituída pelos períodos gerados.
FAULT_PERIODS = [
    ("2024-05-03 11:00", "2024-05-03 11:35"),
    ("2024-06-25 17:20", "2024-08-02 14:00"),
]

# --- Remoção de períodos com o equipamento desligado (None para desativar) ----
OFF_FILTER = dict(
    variable="corrente_motor",  # variável que indica se o equipamento está ligado
    threshold=5,                # considerado desligado quando variable <= threshold
    min_duration=0,             # duração mínima (min) de um desligamento
    pre_cut=30,                 # minutos removidos antes de cada parada (transitório)
    post_cut=30,                # minutos removidos depois de cada partida (transitório)
)

# --- Divisão treino/teste (temporal) -------------------------------------------
TRAIN_PERIOD = ("2024-01-01", "2024-03-05 23:59")
TEST_PERIOD = ("2024-03-06", "2024-03-31 23:59")

# --- Janelamento ---------------------------------------------------------------
WINDOW_SIZE = 60  # amostras por janela (60 min com dados de 1 min)
STEP = 30         # passo entre janelas (50% de sobreposição)

# --- Deep learning -------------------------------------------------------------
DL_MODELS = ["CNN1D", "InceptionTime", "ResNet1D", "ResNet2D"]
EPOCHS = 30
BATCH_SIZE = 128
VAL_FRACTION = 0.2  # fração final do treino usada para validação e escolha do melhor modelo
ARTIFACTS_DIR = Path("../artifacts")  # onde os pesos dos melhores modelos são salvos

RANDOM_STATE = 42
N_JOBS = -1

## 3. Carregamento dos dados

In [ ]:
if USE_SYNTHETIC_DATA:
    df_raw, FAULT_PERIODS = make_synthetic_dataset(seed=RANDOM_STATE)
else:
    df_raw = load_dataset(DATA_PATH, TIMESTAMP_COL, drop_columns=DROP_COLUMNS)

print(f"Dataset: {df_raw.shape[0]} amostras x {df_raw.shape[1] - 1} variáveis")
df_raw.head()

## 4. Pré-processamento e rotulagem

Períodos com o equipamento desligado (e os transitórios de partida/parada) não representam a operação normal nem a falha, então são removidos. Em seguida, cada amostra recebe o rótulo `1` se estiver dentro de um período de falha e `0` caso contrário.

In [ ]:
df = df_raw
if OFF_FILTER is not None:
    df = remove_off_periods(df, timestamp_col=TIMESTAMP_COL, **OFF_FILTER)
    print(f"Removidas {len(df_raw) - len(df)} amostras (equipamento desligado ou em transitório)")

df = label_periods(df, FAULT_PERIODS, TIMESTAMP_COL, LABEL_COL)
FEATURE_COLS = [c for c in df.columns if c not in (TIMESTAMP_COL, LABEL_COL)]

print(f"Variáveis: {FEATURE_COLS}")
print(df[LABEL_COL].value_counts().rename({0: "normal", 1: "falha"}))

### Visualização de uma variável e dos períodos de falha

In [ ]:
plot_signal(df, PLOT_VARIABLE, LABEL_COL, TIMESTAMP_COL)

## 5. Divisão treino/teste e janelamento

A divisão é **temporal** (treino no passado, teste no futuro), como aconteceria em produção. Uma divisão aleatória colocaria janelas quase idênticas nos dois conjuntos e superestimaria o desempenho.

Cada janela vira um exemplo de classificação, rotulado como falha quando mais da metade das amostras é falha. Janelas que atravessam lacunas no tempo (períodos removidos) são descartadas.

In [ ]:
df_train = select_period(df, *TRAIN_PERIOD, timestamp_col=TIMESTAMP_COL)
df_test = select_period(df, *TEST_PERIOD, timestamp_col=TIMESTAMP_COL)

window_args = dict(
    feature_cols=FEATURE_COLS, label_col=LABEL_COL, timestamp_col=TIMESTAMP_COL,
    window_size=WINDOW_SIZE, step=STEP, freq=SAMPLING_FREQ,
)
X_train, y_train, ts_train = make_windows(df_train, **window_args)
X_test, y_test, ts_test = make_windows(df_test, **window_args)

print(f"Treino: {X_train.shape} (janelas, tempo, variáveis) | falhas: {y_train.sum()} ({y_train.mean():.1%})")
print(f"Teste:  {X_test.shape} (janelas, tempo, variáveis) | falhas: {y_test.sum()} ({y_test.mean():.1%})")

# Resultados de todos os modelos no conjunto de teste, para a comparação final
results, test_scores = {}, {}

def register(name, y_pred, y_score):
    results[name] = classification_metrics(y_test, y_pred, y_score)
    test_scores[name] = y_score
    print(pd.Series(results[name]).round(4).to_string())

# Abordagem 1 — Features estatísticas + ML clássico

Cada janela é resumida por estatísticas de cada variável (média, desvio padrão, quantis, assimetria, curtose, energia, autocorrelação etc.). O resultado é uma tabela comum, que pode ser usada por qualquer classificador tabular.

Os hiperparâmetros são escolhidos com `TimeSeriesSplit`, que também respeita a ordem temporal.

In [ ]:
F_train = extract_window_features(X_train, FEATURE_COLS, index=ts_train)
F_test = extract_window_features(X_test, FEATURE_COLS, index=ts_test)

feature_scaler = StandardScaler().fit(F_train)
F_train_scaled = pd.DataFrame(feature_scaler.transform(F_train), columns=F_train.columns, index=F_train.index)
F_test_scaled = pd.DataFrame(feature_scaler.transform(F_test), columns=F_test.columns, index=F_test.index)

tscv = TimeSeriesSplit(n_splits=5)
print("Features por janela:", F_train.shape[1])

## Random Forest

In [ ]:
rf_search = GridSearchCV(
    RandomForestClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=N_JOBS),
    param_grid={"n_estimators": [300, 600], "max_depth": [None, 10]},
    cv=tscv, scoring="average_precision", n_jobs=1,
)
rf_search.fit(F_train_scaled, y_train)
rf_clf = rf_search.best_estimator_
print("Melhores hiperparâmetros:", rf_search.best_params_)

register("RandomForest", rf_clf.predict(F_test_scaled), rf_clf.predict_proba(F_test_scaled)[:, 1])
plot_predictions(ts_test, y_test, test_scores["RandomForest"], "Random Forest — probabilidade de falha vs. real")

### Interpretação com SHAP

Os valores SHAP indicam quanto cada feature empurrou a previsão em direção à falha. Isso ajuda a descobrir quais variáveis (e quais estatísticas delas) caracterizam o problema.

In [ ]:
F_shap = F_test_scaled.iloc[: min(500, len(F_test_scaled))]
shap_values = shap.TreeExplainer(rf_clf).shap_values(F_shap)
shap_values = shap_values[:, :, 1] if np.ndim(shap_values) == 3 else shap_values  # classe "falha"

shap.summary_plot(shap_values, F_shap, plot_type="bar", max_display=15)
shap.summary_plot(shap_values, F_shap, max_display=15)

## XGBoost

In [ ]:
xgb_search = RandomizedSearchCV(
    XGBClassifier(
        eval_metric="aucpr",
        scale_pos_weight=(y_train == 0).sum() / max((y_train == 1).sum(), 1),
        random_state=RANDOM_STATE,
        n_jobs=N_JOBS,
    ),
    param_distributions={
        "n_estimators": [100, 300, 500, 800],
        "learning_rate": [0.01, 0.05, 0.1, 0.2],
        "max_depth": [3, 4, 5, 6, 8],
        "subsample": [0.7, 0.8, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.8, 0.9, 1.0],
        "gamma": [0, 0.1, 0.2],
    },
    n_iter=10, cv=tscv, scoring="average_precision", random_state=RANDOM_STATE, n_jobs=1,
)
xgb_search.fit(F_train_scaled, y_train)
xgb_clf = xgb_search.best_estimator_
print("Melhores hiperparâmetros:", xgb_search.best_params_)

register("XGBoost", xgb_clf.predict(F_test_scaled), xgb_clf.predict_proba(F_test_scaled)[:, 1])
plot_predictions(ts_test, y_test, test_scores["XGBoost"], "XGBoost — probabilidade de falha vs. real")

## kNN

In [ ]:
knn_search = GridSearchCV(
    KNeighborsClassifier(n_jobs=N_JOBS),
    param_grid={"n_neighbors": list(range(1, 20, 2)), "weights": ["uniform", "distance"]},
    cv=tscv, scoring="average_precision", n_jobs=1,
)
knn_search.fit(F_train_scaled, y_train)
knn_clf = knn_search.best_estimator_
print("Melhores hiperparâmetros:", knn_search.best_params_)

register("kNN", knn_clf.predict(F_test_scaled), knn_clf.predict_proba(F_test_scaled)[:, 1])
plot_predictions(ts_test, y_test, test_scores["kNN"], "kNN — probabilidade de falha vs. real")

# Abordagem 2 — ROCKET e MiniROCKET

[ROCKET](https://arxiv.org/abs/1910.13051) aplica milhares de kernels convolucionais **aleatórios** a cada janela e usa estatísticas simples das saídas como features. Um classificador linear (`RidgeClassifierCV`, recomendado pelos autores) é treinado sobre elas. É rápido e costuma ser um baseline muito forte. O MiniROCKET é uma variante quase determinística e ainda mais rápida.

As janelas são padronizadas com média e desvio do treino (por variável), e o sktime espera os dados no formato `(janelas, variáveis, tempo)`.

In [ ]:
window_scaler = fit_window_scaler(X_train)
X_train_panel = scale_windows(window_scaler, X_train).transpose(0, 2, 1)
X_test_panel = scale_windows(window_scaler, X_test).transpose(0, 2, 1)

for name, transformer in {
    # normalise=False: as janelas já estão padronizadas pelo scaler do treino; normalizar cada janela
    # individualmente apagaria as mudanças de nível que caracterizam a falha
    "ROCKET": Rocket(num_kernels=10000, normalise=False, random_state=RANDOM_STATE, n_jobs=N_JOBS),
    "MiniROCKET": MiniRocketMultivariate(num_kernels=10000, random_state=RANDOM_STATE, n_jobs=N_JOBS),
}.items():
    Z_train = transformer.fit_transform(X_train_panel)
    Z_test = transformer.transform(X_test_panel)

    ridge = RidgeClassifierCV(alphas=np.logspace(-5, 5, 100), class_weight="balanced").fit(Z_train, y_train)
    print(f"--- {name} ({Z_train.shape[1]} features) ---")
    register(name, ridge.predict(Z_test), ridge.decision_function(Z_test))
    plot_predictions(ts_test, y_test, ridge.decision_function(Z_test), f"{name} — score vs. real")

# Abordagem 3 — Deep learning (PyTorch)

As redes recebem a janela normalizada inteira e aprendem as features sozinhas. Todas usam:

- **Focal loss** com **pesos de classe balanceados**, para lidar com o desbalanceamento;
- **AdamW** e redução da taxa de aprendizado quando a loss de validação estagna;
- **Checkpoint** do melhor modelo pela AUC na validação.

A validação é o trecho final do período de treino (`VAL_FRACTION`), de modo que o conjunto de teste não influencia nenhuma escolha. Os pesos do melhor modelo de cada arquitetura são salvos em `ARTIFACTS_DIR`; o modelo da última época continua disponível em `dl_results[nome].last_model`.

In [ ]:
# Algumas janelas entre treino e validação são descartadas para evitar sobreposição
gap = WINDOW_SIZE // STEP
X_fit, X_val, y_fit, y_val = temporal_split(X_train, y_train, val_fraction=VAL_FRACTION, gap=gap)

dl_scaler = fit_window_scaler(X_fit)
X_fit_s, X_val_s, X_test_s = (scale_windows(dl_scaler, X) for X in (X_fit, X_val, X_test))
print(f"Ajuste: {X_fit.shape} | validação: {X_val.shape} (falhas: {y_val.sum()}) | teste: {X_test.shape}")

dl_results = {}
for name in DL_MODELS:
    print(f"\n===== {name} =====")
    dl_results[name] = train_model(
        MODELS[name](n_features=len(FEATURE_COLS)),
        X_fit_s, y_fit, X_val_s, y_val,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        weight_decay=1e-4 if name == "CNN1D" else 0.0,
        checkpoint_path=ARTIFACTS_DIR / f"best_{name.lower()}.pt",
        seed=RANDOM_STATE,
    )

In [ ]:
for name, result in dl_results.items():
    plot_history(result.history, title=f"{name} (melhor época: {result.best_epoch})")

    y_score = predict_proba(result.best_model, X_test_s)
    print(f"--- {name} ---")
    register(name, (y_score >= 0.5).astype(int), y_score)
    plot_predictions(ts_test, y_test, y_score, f"{name} — probabilidade de falha vs. real")

# Comparação dos modelos

Todas as abordagens usam exatamente as mesmas janelas de teste, então as métricas são diretamente comparáveis.

- **recall**: fração das janelas de falha detectadas;
- **precision**: fração dos alarmes que eram falhas de fato;
- **pr_auc** (average precision) e **roc_auc**: qualidade do ranqueamento, independente do limiar de decisão. Com classes muito desbalanceadas, a PR AUC é a mais informativa.

In [ ]:
pd.DataFrame(results).T.sort_values("pr_auc", ascending=False).round(4)

In [ ]:
plot_roc_curves(y_test, test_scores, title="Curvas ROC no conjunto de teste")

## Próximos passos

- Ajustar o **limiar de decisão** de cada modelo conforme o custo de falsos alarmes vs. falhas não detectadas;
- Testar outros tamanhos de janela (`WINDOW_SIZE`) e passos (`STEP`);
- Para colocar um modelo em produção, aplicar o mesmo pré-processamento, janelamento e normalização (scaler ajustado no treino) aos dados novos antes de chamar `predict`/`predict_proba`.